# MongoDB Fundamentals

## Notebook Objectives

This notebook introduces the fundamentals of MongoDB using both **PyMongo** and **MongoDB Shell (mongosh)**.

Throughout this notebook, every MongoDB operation will be demonstrated using:

- **PyMongo** – The official Python driver for MongoDB.
- **MongoDB Shell (mongosh)** – The command-line interface used to interact with MongoDB.

By the end of this notebook, you will be able to:

- Connect to a MongoDB server.
- Create databases and collections.
- Insert, retrieve, update, and delete documents.
- Use MongoDB query operators.
- Perform aggregation pipeline operations.

> **Note:** For operations that modify the database (Insert, Update, Delete), the **mongosh** commands are provided as commented code to prevent duplicate execution. You can copy and execute them directly in the MongoDB Shell whenever required.

# Import Required Libraries

The following Python libraries will be used throughout this notebook.

| Library | Purpose |
|----------|----------|
| `pymongo` | Connect Python applications with MongoDB |
| `pandas` | Data manipulation and analysis |
| `numpy` | Numerical computations |
| `matplotlib` | Data visualization |
| `seaborn` | Statistical visualization |

In [ ]:
# =====================================
# Import Required Libraries
# =====================================

from pymongo import MongoClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

print("All required libraries imported successfully.")

All required libraries imported successfully.


# Connect to MongoDB Server

Before performing any database operations, we must establish a connection with the MongoDB server.

The default connection URL is:

```text
mongodb://localhost:27017
```

where

- **localhost** → MongoDB Server
- **27017** → Default MongoDB Port

After creating the client, we will verify that the MongoDB server is running successfully.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

client = MongoClient("mongodb://localhost:27017")

try:
    client.admin.command("ping")
    print("Successfully connected to MongoDB Server.")
except Exception as e:
    print("Connection Failed")
    print(e)

Successfully connected to MongoDB Server.


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.adminCommand({ ping: 1 })"

{ ok: 1 }


# Display Available Databases

MongoDB stores multiple databases on a single server.

The following commands display all currently available databases.

> **Note:** MongoDB displays only those databases that contain at least one collection with data.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'local']

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin    40.00 KiB
config  108.00 KiB
local    64.00 KiB


# Database

A **database** is a logical container that stores one or more **collections**.

Just as a folder on your computer can contain multiple files, a MongoDB database can contain multiple collections.

In this notebook, we will create a database named **ecommerce**.

> **Important:**  
> In MongoDB, simply selecting a database **does not create it**. The database is physically created **only after a document is inserted into one of its collections**.

For now, we will simply select the database. It will appear in MongoDB only after data is inserted in later sections.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

db = client["ecommerce"]

db

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'ecommerce')

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce"

switched to db ecommerce


# Understanding the Database Object

The variable **`db`** is a **Database object** returned by PyMongo.

It represents the selected MongoDB database and allows us to perform operations such as:

- Creating collections
- Listing collections
- Inserting documents
- Querying data
- Updating documents
- Deleting documents

Since no documents have been inserted yet, the **ecommerce** database does not physically exist on the MongoDB server.

You can verify this by listing all available databases.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'local']

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin    40.00 KiB
config  108.00 KiB
local    64.00 KiB


# Collection

A **collection** is a group of related documents stored within a MongoDB database.

It is similar to a **table** in a relational database (RDBMS), but unlike a table:

- Collections do **not** require a fixed schema.
- Documents in the same collection can have different fields.
- MongoDB automatically creates a collection when the first document is inserted into it.

For better understanding, consider the following comparison:

| Relational Database | MongoDB |
|---------------------|----------|
| Database | Database |
| Table | Collection |
| Row | Document |
| Column | Field |

In this notebook, we will use a collection named **customers** to store customer information for our e-commerce application.

> **Important:**  
> Like databases, selecting a collection does **not** physically create it. The collection is created automatically when the first document is inserted.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customers = db["customers"]

customers

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'ecommerce'), 'customers')

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.getCollection('customers')"

switched to db ecommerce;


# Understanding the Collection Object

The variable **customers** is a **Collection object**.

It represents the **customers** collection inside the **ecommerce** database.

Using this object, we can perform operations such as:

- Insert documents
- Retrieve documents
- Update documents
- Delete documents
- Aggregate data

Since no documents have been inserted yet, the **customers** collection has not been created physically.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

db.list_collection_names()

[]

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; show collections"

switched to db ecommerce;


# Document

A **document** is the basic unit of data storage in MongoDB.

A document is a collection of **field-value pairs**, similar to a JSON object.

For example, a customer in an e-commerce application can be represented as a single document.

```json
{
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "gender": "Male",
    "age": 28,
    "email": "rahul@example.com",
    "city": "Hyderabad",
    "membership": "Gold",
    "is_active": true
}
```

Each field stores a specific piece of information about the customer.

Unlike relational databases, MongoDB documents can have different fields. This flexible schema makes MongoDB suitable for applications where the data structure changes frequently.

# JSON vs BSON

Although MongoDB documents look like **JSON**, MongoDB actually stores them internally as **BSON (Binary JSON)**.

BSON extends JSON by supporting additional data types and enabling faster storage and retrieval.

| JSON | BSON |
|------|------|
| Text-based format | Binary format |
| Human-readable | Machine-efficient |
| Limited data types | Supports additional data types such as ObjectId, Date, Binary, Decimal128, etc. |
| Used for data interchange | Used for internal storage by MongoDB |

When we write documents in Python or in the MongoDB Shell, they appear like JSON, but MongoDB automatically converts them into BSON before storing them.

# The `_id` Field and ObjectId

Every document stored in MongoDB must contain a unique field named **`_id`**.

If we do not provide an `_id`, MongoDB automatically generates one.

The automatically generated `_id` is usually of type **ObjectId**.

Example:

```text
ObjectId("6878fd5f5d5d4e8c2f7c3b9a")
```

The `_id` field serves as the **primary key** of a MongoDB document.

Its purpose is to uniquely identify every document within a collection.

> **Note:**  
> During our first insert operation, we will not specify the `_id` field. MongoDB will generate it automatically.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

from bson import ObjectId

# Generate a sample ObjectId (for demonstration)

sample_id = ObjectId()

sample_id

ObjectId('6a5cb76683106d9367c89b86')

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "ObjectId()"

ObjectId('6a5cb79992d204f8f4d7a6c1')


# Insert a Single Document

The **`insert_one()`** method is used to insert a single document into a MongoDB collection.

If the specified database or collection does not already exist, MongoDB automatically creates them when the first document is inserted.

Since we have not yet inserted any data, this operation will:

- Create the **ecommerce** database.
- Create the **customers** collection.
- Insert one customer document.
- Automatically generate an **`_id`** field because we are not providing one.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customer = {
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "gender": "Male",
    "age": 28,
    "email": "rahul.sharma@example.com",
    "phone": "9876543210",
    "city": "Hyderabad",
    "state": "Telangana",
    "membership": "Gold",
    "join_date": "2025-01-15",
    "is_active": True
}

result = customers.insert_one(customer)

print("Inserted Document ID:", result.inserted_id)

Inserted Document ID: 6a5cb7de83106d9367c89b87


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval '
# use ecommerce
# db.customers.insertOne({
#     customer_id: 1001,
#     name: "Rahul Sharma",
#     gender: "Male",
#     age: 28,
#     email: "rahul.sharma@example.com",
#     phone: "9876543210",
#     city: "Hyderabad",
#     state: "Telangana",
#     membership: "Gold",
#     join_date: "2025-01-15",
#     is_active: true
# })'

# Understanding the Return Value

The `insert_one()` method returns an **InsertOneResult** object.

The most commonly used attribute is:

- **`inserted_id`** – Returns the `_id` of the newly inserted document.

This value can be stored and used later to retrieve, update, or delete the same document.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

type(result)

pymongo.results.InsertOneResult

In [ ]:
# =====================================
# PyMongo Command
# =====================================

result.inserted_id

ObjectId('6a5cb7de83106d9367c89b87')

# Verify Database and Collection Creation

Earlier, the **ecommerce** database and **customers** collection did not physically exist.

After inserting the first document, MongoDB automatically creates both.

Let's verify this.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'ecommerce', 'local']

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin       40.00 KiB
config     108.00 KiB
ecommerce   40.00 KiB
local       64.00 KiB


In [ ]:
# =====================================
# PyMongo Command
# =====================================

db.list_collection_names()

['customers']

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; show collections"

switched to db ecommerce;


# Retrieve a Single Document

The **`find_one()`** method retrieves the **first document** that matches the specified query.

If no query is provided, MongoDB returns the **first document** in the collection.

The returned value is a Python dictionary (`dict`) representing a MongoDB document.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customers.find_one()

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'customer_id': 1001,
 'name': 'Rahul Sharma',
 'gender': 'Male',
 'age': 28,
 'email': 'rahul.sharma@example.com',
 'phone': '9876543210',
 'city': 'Hyderabad',
 'state': 'Telangana',
 'membership': 'Gold',
 'join_date': '2025-01-15',
 'is_active': True}

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.customers.findOne()"

switched to db ecommerce;


# Retrieve All Documents

The **`find()`** method retrieves **all documents** that match the specified query.

If no query is provided, every document in the collection is returned.

Unlike `find_one()`, the `find()` method returns a **Cursor object**.

A cursor is an iterator that allows MongoDB to efficiently return documents one at a time instead of loading all documents into memory.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customers.find()

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.customers.find()"

switched to db ecommerce;


# Display Documents Using a Loop

Since the `find()` method returns a **Cursor**, we usually iterate through it using a loop.

Each iteration returns one document from the collection.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

for customer in customers.find():
    print(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'), 'customer_id': 1001, 'name': 'Rahul Sharma', 'gender': 'Male', 'age': 28, 'email': 'rahul.sharma@example.com', 'phone': '9876543210', 'city': 'Hyderabad', 'state': 'Telangana', 'membership': 'Gold', 'join_date': '2025-01-15', 'is_active': True}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find()"

[
  {
    _id: ObjectId('6a5cb7de83106d9367c89b87'),
    customer_id: 1001,
    name: 'Rahul Sharma',
    gender: 'Male',
    age: 28,
    email: 'rahul.sharma@example.com',
    phone: '9876543210',
    city: 'Hyderabad',
    state: 'Telangana',
    membership: 'Gold',
    join_date: '2025-01-15',
    is_active: true
  }
]


# Count the Number of Documents

The **`count_documents()`** method returns the number of documents that match a specified filter.

Passing an empty filter `{}` counts every document in the collection.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customers.count_documents({})

1

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.countDocuments({})"

1


# Insert Multiple Documents

The **`insert_many()`** method inserts multiple documents into a collection in a single operation.

This method accepts a **list of dictionaries**, where each dictionary represents one document.

Advantages of using `insert_many()`:

- Inserts multiple documents efficiently.
- Reduces communication with the MongoDB server.
- Returns the `_id` values of all inserted documents.

In this section, we will populate our **customers** collection with additional customer records for future examples.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customers_list = [
    {
        "customer_id": 1002,
        "name": "Priya Reddy",
        "gender": "Female",
        "age": 25,
        "email": "priya.reddy@example.com",
        "phone": "9876543211",
        "city": "Hyderabad",
        "state": "Telangana",
        "membership": "Silver",
        "join_date": "2025-02-10",
        "is_active": True
    },
    {
        "customer_id": 1003,
        "name": "Arjun Kumar",
        "gender": "Male",
        "age": 32,
        "email": "arjun.kumar@example.com",
        "phone": "9876543212",
        "city": "Bengaluru",
        "state": "Karnataka",
        "membership": "Gold",
        "join_date": "2024-12-20",
        "is_active": True
    },
    {
        "customer_id": 1004,
        "name": "Sneha Patel",
        "gender": "Female",
        "age": 29,
        "email": "sneha.patel@example.com",
        "phone": "9876543213",
        "city": "Mumbai",
        "state": "Maharashtra",
        "membership": "Platinum",
        "join_date": "2025-03-05",
        "is_active": True
    },
    {
        "customer_id": 1005,
        "name": "Vikram Singh",
        "gender": "Male",
        "age": 35,
        "email": "vikram.singh@example.com",
        "phone": "9876543214",
        "city": "Chennai",
        "state": "Tamil Nadu",
        "membership": "Silver",
        "join_date": "2024-11-18",
        "is_active": False
    },
    {
        "customer_id": 1006,
        "name": "Ananya Das",
        "gender": "Female",
        "age": 27,
        "email": "ananya.das@example.com",
        "phone": "9876543215",
        "city": "Kolkata",
        "state": "West Bengal",
        "membership": "Gold",
        "join_date": "2025-01-28",
        "is_active": True
    },
    {
        "customer_id": 1007,
        "name": "Rohit Verma",
        "gender": "Male",
        "age": 31,
        "email": "rohit.verma@example.com",
        "phone": "9876543216",
        "city": "Pune",
        "state": "Maharashtra",
        "membership": "Bronze",
        "join_date": "2025-04-01",
        "is_active": True
    },
    {
        "customer_id": 1008,
        "name": "Kavya Nair",
        "gender": "Female",
        "age": 26,
        "email": "kavya.nair@example.com",
        "phone": "9876543217",
        "city": "Kochi",
        "state": "Kerala",
        "membership": "Silver",
        "join_date": "2025-02-15",
        "is_active": False
    },
    {
        "customer_id": 1009,
        "name": "Amit Joshi",
        "gender": "Male",
        "age": 38,
        "email": "amit.joshi@example.com",
        "phone": "9876543218",
        "city": "Jaipur",
        "state": "Rajasthan",
        "membership": "Gold",
        "join_date": "2024-10-12",
        "is_active": True
    },
    {
        "customer_id": 1010,
        "name": "Neha Sharma",
        "gender": "Female",
        "age": 24,
        "email": "neha.sharma@example.com",
        "phone": "9876543219",
        "city": "Delhi",
        "state": "Delhi",
        "membership": "Bronze",
        "join_date": "2025-05-08",
        "is_active": True
    },
    {
        "customer_id": 1011,
        "name": "Suresh Iyer",
        "gender": "Male",
        "age": 40,
        "email": "suresh.iyer@example.com",
        "phone": "9876543220",
        "city": "Chennai",
        "state": "Tamil Nadu",
        "membership": "Platinum",
        "join_date": "2024-09-25",
        "is_active": True
    }
]

result = customers.insert_many(customers_list)

print("Inserted", len(result.inserted_ids), "documents.")

Inserted 10 documents.


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.insertMany([
#     { customer_id:1002, name:'Priya Reddy', gender:'Female', age:25, email:'priya.reddy@example.com', phone:'9876543211', city:'Hyderabad', state:'Telangana', membership:'Silver', join_date:'2025-02-10', is_active:true },
#     { customer_id:1003, name:'Arjun Kumar', gender:'Male', age:32, email:'arjun.kumar@example.com', phone:'9876543212', city:'Bengaluru', state:'Karnataka', membership:'Gold', join_date:'2024-12-20', is_active:true }
#     // Remaining documents omitted for readability.
# ])
# "

# Verify the Inserted Documents

After inserting multiple documents, let's verify the total number of documents available in the **customers** collection.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customers.count_documents({})

11

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.countDocuments({})"

11


In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": "Gold"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:'Gold'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  emai

# Query Documents Using Filters

In the previous section, we retrieved all documents from the collection.

However, in real-world applications, we rarely need every document. Instead, we retrieve only those documents that satisfy a specific condition.

MongoDB uses a **query filter** to specify the search criteria.

A query filter is written as a Python dictionary (or a JSON document in the MongoDB Shell).

General Syntax:

```python
collection.find(filter)
```

If the filter is empty (`{}`), all documents are returned.

If the filter contains conditions, only the matching documents are returned.

# Equality Query

The simplest type of query is an **equality query**.

An equality query returns all documents where the specified field exactly matches the given value.

For example, the following query retrieves all customers whose city is **Hyderabad**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE city = 'Hyderabad';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"city": "Hyderabad"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({city:'Hyderabad'}).pretty()"

[
  {
    _id: ObjectId('6a5cb7de83106d9367c89b87'),
    customer_id: 1001,
    name: 'Rahul Sharma',
    gender: 'Male',
    age: 28,
    email: 'rahul.sharma@example.com',
    phone: '9876543210',
    city: 'Hyderabad',
    state: 'Telangana',
    membership: 'Gold',
    join_date: '2025-01-15',
    is_active: true
  },
  {
    _id: ObjectId('6a5cba4483106d9367c89b88'),
    customer_id: 1002,
    name: 'Priya Reddy',
    gender: 'Female',
    age: 25,
    email: 'priya.reddy@example.com',
    phone: '9876543211',
    city: 'Hyderabad',
    state: 'Telangana',
    membership: 'Silver',
    join_date: '2025-02-10',
    is_active: true
  }
]


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({city:'Hyderabad'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}


# Another Equality Query

We can filter documents using any field present in the document.

The following example retrieves all customers having a **Gold** membership.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership = 'Gold';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": "Gold"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:'Gold'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  emai

# Comparison Query Operators

Comparison operators are used to retrieve documents based on numerical or alphabetical comparisons.

These operators are commonly used for filtering data such as:

- Customers older than a certain age
- Products with prices above a specific value
- Employees with salaries below a threshold

The following comparison operators are supported by MongoDB.

| Operator | Description | SQL Equivalent |
|----------|-------------|----------------|
| `$gt` | Greater Than | `>` |
| `$gte` | Greater Than or Equal To | `>=` |
| `$lt` | Less Than | `<` |
| `$lte` | Less Than or Equal To | `<=` |
| `$ne` | Not Equal To | `!=` |

# Greater Than (`$gt`)

The **`$gt`** operator retrieves documents where the field value is **greater than** the specified value.

The following example retrieves customers whose age is greater than **30**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age > 30;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$gt": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$gt:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  ema

# Greater Than or Equal To (`$gte`)

The **`$gte`** operator retrieves documents where the field value is **greater than or equal to** the specified value.

The following example retrieves customers whose age is **30 years or older**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age >= 30;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$gte": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$gte:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  ema

# Less Than (`$lt`)

The **`$lt`** operator retrieves documents where the field value is **less than** the specified value.

The following example retrieves customers whose age is less than **30**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age < 30;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$lt": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$lt:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age:

# Less Than or Equal To (`$lte`)

The **`$lte`** operator retrieves documents where the field value is **less than or equal to** the specified value.

The following example retrieves customers whose age is **30 years or younger**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE age <= 30;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"age": {"$lte": 30}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$lte:30}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age:

# Not Equal To (`$ne`)

The **`$ne`** operator retrieves documents where the field value is **not equal to** the specified value.

The following example retrieves customers whose membership is **not Gold**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership != 'Gold';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": {"$ne": "Gold"}})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_i

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:{$ne:'Gold'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age

# Logical Query Operators

Logical operators combine multiple query conditions into a single query.

They are useful when data must satisfy one or more conditions simultaneously.

MongoDB provides the following logical operators:

| Operator | Description | SQL Equivalent |
|----------|-------------|----------------|
| `$and` | All conditions must be true | `AND` |
| `$or` | At least one condition must be true | `OR` |
| `$not` | Negates a condition | `NOT` |
| `$nor` | None of the conditions should be true | `NOT (A OR B)` |

# AND Operator (`$and`)

The **`$and`** operator returns documents only when **all specified conditions are true**.

The following example retrieves customers who:

- belong to **Gold** membership, and
- are older than **30** years.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership = 'Gold'
AND age > 30;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "$and": [
        {"membership": "Gold"},
        {"age": {"$gt": 30}}
    ]
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,
 'email': 'amit.joshi@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-10-12',
 'membership': 'Gold',
 'name': 'Amit Joshi',
 'phone': '9876543218',
 'state': 'Rajasthan'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({$and:[{membership:'Gold'},{age:{$gt:30}}]}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  email: 'amit.joshi@example.com',
  phone: '9876543218',
  city: 'Jaipur',
  state: 'Rajasthan',
  membership: 'Gold',
  join_date: '2024-10-12',
  is_active: true
}


# OR Operator (`$or`)

The **`$or`** operator returns documents when **at least one** of the specified conditions is true.

The following example retrieves customers who are from:

- Hyderabad, or
- Chennai.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE city = 'Hyderabad'
OR city = 'Chennai';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "$or": [
        {"city": "Hyderabad"},
        {"city": "Chennai"}
    ]
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b91'),
 'age': 40,
 'city': 'Chennai',
 'customer_i

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({$or:[{city:'Hyderabad'},{city:'Chennai'}]}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b91'),
  customer_id: 1011,
  name: 'Suresh Iyer',
  gender: 'Male',
  age: 4

# NOT Operator (`$not`)

The **`$not`** operator negates a query condition.

The following example retrieves customers whose age is **not greater than 30**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE NOT(age > 30);
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "age": {
        "$not": {
            "$gt": 30
        }
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$not:{$gt:30}}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age:

# NOR Operator (`$nor`)

The **`$nor`** operator returns documents only when **none** of the specified conditions are true.

The following example retrieves customers who are:

- not from Hyderabad, and
- not from Chennai.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE city <> 'Hyderabad'
AND city <> 'Chennai';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "$nor": [
        {"city": "Hyderabad"},
        {"city": "Chennai"}
    ]
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({$nor:[{city:'Hyderabad'},{city:'Chennai'}]}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  

# Element and Evaluation Query Operators

Element and Evaluation operators allow us to query documents based on:

- Whether a field exists
- The data type of a field
- Whether a field belongs to a list of values
- Whether a field does not belong to a list of values
- Whether a field matches a text pattern

These operators are widely used in real-world applications for searching and filtering data.

# Exists Operator (`$exists`)

The **`$exists`** operator checks whether a particular field exists in a document.

It does **not** check the field's value—it only verifies the presence or absence of the field.

The following example retrieves customers that contain the **phone** field.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "phone": {
        "$exists": True
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({phone:{$exists:true}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,

# In Operator (`$in`)

The **`$in`** operator retrieves documents where a field matches **any one** of the values specified in a list.

The following example retrieves customers whose membership is either:

- Gold
- Platinum

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership IN ('Gold', 'Platinum');
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "membership": {
        "$in": ["Gold", "Platinum"]
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id':

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:{$in:['Gold','Platinum']}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,

# Not In Operator (`$nin`)

The **`$nin`** operator retrieves documents whose field value is **not present** in the specified list.

The following example retrieves customers whose membership is neither **Gold** nor **Platinum**.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership NOT IN ('Gold', 'Platinum');
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "membership": {
        "$nin": ["Gold", "Platinum"]
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8e'),
 'age': 26,
 'city': 'Kochi',
 'customer_id': 1

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:{$nin:['Gold','Platinum']}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8e'),
  customer_id: 1008,
  name: 'Kavya Nair',
  gender: 'Female',
  age: 26,

# Regular Expression Operator (`$regex`)

The **`$regex`** operator performs pattern matching on string fields.

It is useful for searching names, email addresses, cities, and other text-based fields.

The following example retrieves customers whose names start with the letter **A**.

Equivalent SQL query (conceptually):

```sql
SELECT *
FROM customers
WHERE name LIKE 'A%';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "name": {
        "$regex": "^A"
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,
 'email': 'amit.joshi@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-10-12',
 'membership': 'Gold',
 'name': 'Amit Joshi',
 'phone': '9876543218',
 'state': 'Rajasthan'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({name:{$regex:'^A'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  email: 'amit.joshi@example.com',
  phone: '9876543218',
  city: 'Jaipur',
  state: 'Rajasthan',
  membership: 'Gold',
  join_date: '2024-10-12',
  is_active: true
}


# Type Operator (`$type`)

The **`$type`** operator retrieves documents where a field is of a specific BSON data type.

This is useful for validating data consistency in a collection.

The following example retrieves documents where the **age** field is stored as an integer.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({
    "age": {
        "$type": "int"
    }
})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({age:{$type:'int'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,

# Projection

By default, MongoDB returns **all fields** of every matching document.

However, in many situations, we need only a few fields instead of the complete document.

**Projection** is used to specify which fields should be included or excluded in the query result.

General Syntax:

```python
collection.find(filter, projection)
```

Equivalent SQL query:

```sql
SELECT column1, column2
FROM table_name;
```

Using projection improves query performance by returning only the required data.

# Include Specific Fields

To include specific fields in the query result, assign the value **1** to those fields in the projection document.

By default, MongoDB also returns the **`_id`** field unless it is explicitly excluded.

The following example displays only the customer's **name**, **city**, and **membership**.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find(
    {},
    {
        "name": 1,
        "city": 1,
        "membership": 1
    }
)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'city': 'Hyderabad',
 'membership': 'Gold',
 'name': 'Rahul Sharma'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'city': 'Hyderabad',
 'membership': 'Silver',
 'name': 'Priya Reddy'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'city': 'Bengaluru',
 'membership': 'Gold',
 'name': 'Arjun Kumar'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'city': 'Mumbai',
 'membership': 'Platinum',
 'name': 'Sneha Patel'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'city': 'Chennai',
 'membership': 'Silver',
 'name': 'Vikram Singh'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'city': 'Kolkata',
 'membership': 'Gold',
 'name': 'Ananya Das'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'city': 'Pune',
 'membership': 'Bronze',
 'name': 'Rohit Verma'}
{'_id': ObjectId('6a5cba4483106d9367c89b8e'),
 'city': 'Kochi',
 'membership': 'Silver',
 'name': 'Kavya Nair'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'city': 'Jaipur',
 'membership': 'Gol

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {name:1, city:1, membership:1}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  name: 'Rahul Sharma',
  city: 'Hyderabad',
  membership: 'Gold'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  name: 'Priya Reddy',
  city: 'Hyderabad',
  membership: 'Silver'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  name: 'Arjun Kumar',
  city: 'Bengaluru',
  membership: 'Gold'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  name: 'Sneha Patel',
  city: 'Mumbai',
  membership: 'Platinum'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  name: 'Vikram Singh',
  city: 'Chennai',
  membership: 'Silver'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  name: 'Ananya Das',
  city: 'Kolkata',
  membership: 'Gold'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  name: 'Rohit Verma',
  city: 'Pune',
  membership: 'Bronze'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8e'),
  name: 'Kavya Nair',
  city: 'Kochi',
  membership: 'Silver'
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  name: 'Amit Joshi',
  city: 'Jaipur',
  mem

# Exclude the `_id` Field

The `_id` field is included automatically in every query result.

If it is not required, it can be excluded by assigning **0** to the `_id` field.

The following example displays only the customer's **name**, **city**, and **membership**, without the `_id`.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find(
    {},
    {
        "_id": 0,
        "name": 1,
        "city": 1,
        "membership": 1
    }
)

for customer in cursor:
    pprint(customer)

{'city': 'Hyderabad', 'membership': 'Gold', 'name': 'Rahul Sharma'}
{'city': 'Hyderabad', 'membership': 'Silver', 'name': 'Priya Reddy'}
{'city': 'Bengaluru', 'membership': 'Gold', 'name': 'Arjun Kumar'}
{'city': 'Mumbai', 'membership': 'Platinum', 'name': 'Sneha Patel'}
{'city': 'Chennai', 'membership': 'Silver', 'name': 'Vikram Singh'}
{'city': 'Kolkata', 'membership': 'Gold', 'name': 'Ananya Das'}
{'city': 'Pune', 'membership': 'Bronze', 'name': 'Rohit Verma'}
{'city': 'Kochi', 'membership': 'Silver', 'name': 'Kavya Nair'}
{'city': 'Jaipur', 'membership': 'Gold', 'name': 'Amit Joshi'}
{'city': 'Delhi', 'membership': 'Bronze', 'name': 'Neha Sharma'}
{'city': 'Chennai', 'membership': 'Platinum', 'name': 'Suresh Iyer'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {_id:0, name:1, city:1, membership:1}).forEach(doc => printjson(doc))"

{
  name: 'Rahul Sharma',
  city: 'Hyderabad',
  membership: 'Gold'
}
{
  name: 'Priya Reddy',
  city: 'Hyderabad',
  membership: 'Silver'
}
{
  name: 'Arjun Kumar',
  city: 'Bengaluru',
  membership: 'Gold'
}
{
  name: 'Sneha Patel',
  city: 'Mumbai',
  membership: 'Platinum'
}
{
  name: 'Vikram Singh',
  city: 'Chennai',
  membership: 'Silver'
}
{
  name: 'Ananya Das',
  city: 'Kolkata',
  membership: 'Gold'
}
{
  name: 'Rohit Verma',
  city: 'Pune',
  membership: 'Bronze'
}
{
  name: 'Kavya Nair',
  city: 'Kochi',
  membership: 'Silver'
}
{
  name: 'Amit Joshi',
  city: 'Jaipur',
  membership: 'Gold'
}
{
  name: 'Neha Sharma',
  city: 'Delhi',
  membership: 'Bronze'
}
{
  name: 'Suresh Iyer',
  city: 'Chennai',
  membership: 'Platinum'
}


# Exclude Specific Fields

Instead of specifying the required fields, we can exclude unnecessary fields by assigning **0** to them.

> **Important:**  
> MongoDB does **not** allow mixing inclusion (`1`) and exclusion (`0`) in the same projection document, except for the `_id` field.

The following example excludes the **email** and **phone** fields.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find(
    {},
    {
        "email": 0,
        "phone": 0
    }
)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d93

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {email:0, phone:0}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 

# Invalid Projection

MongoDB does **not** allow mixing field inclusion and exclusion in the same projection document (except for `_id`).

For example, the following query is invalid because it attempts to include **name** while excluding **city**.

This query raises an error.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

try:
    cursor = customers.find(
        {},
        {
            "name": 1,
            "city": 0
        }
    )

    for customer in cursor:
        pprint(customer)

except Exception as e:
    print(e)

Cannot do exclusion on field city in inclusion projection, full error: {'ok': 0.0, 'errmsg': 'Cannot do exclusion on field city in inclusion projection', 'code': 31254, 'codeName': 'Location31254'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "db.getSiblingDB('ecommerce').customers.find({}, {name:1, city:0})"

# Sorting Query Results

By default, MongoDB returns documents in their natural storage order.

The **`sort()`** method is used to arrange the query results in a specific order.

General Syntax:

```python
collection.find(filter).sort(field, order)
```

where:

- **1** → Ascending Order
- **-1** → Descending Order

Equivalent SQL query:

```sql
SELECT *
FROM customers
ORDER BY age ASC;
```

# Sort in Ascending Order

To sort documents in **ascending order**, specify **1** as the sorting order.

The following example displays all customers sorted by **age** from the youngest to the oldest.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().sort("age", 1)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b90'),
 'age': 24,
 'city': 'Delhi',
 'customer_id': 1010,
 'email': 'neha.sharma@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-05-08',
 'membership': 'Bronze',
 'name': 'Neha Sharma',
 'phone': '9876543219',
 'state': 'Delhi'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b8e'),
 'age': 26,
 'city': 'Kochi',
 'customer_id': 1008,
 'email': 'kavya.nair@example.com',
 'gender': 'Female',
 'is_active': False,
 'join_date': '2025-02-15',
 'membership': 'Silver',
 'name': 'Kavya Nair',
 'phone': '9876543217',
 'state': 'Kerala'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'em

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().sort({age:1}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b90'),
  customer_id: 1010,
  name: 'Neha Sharma',
  gender: 'Female',
  age: 24,
  email: 'neha.sharma@example.com',
  phone: '9876543219',
  city: 'Delhi',
  state: 'Delhi',
  membership: 'Bronze',
  join_date: '2025-05-08',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8e'),
  customer_id: 1008,
  name: 'Kavya Nair',
  gender: 'Female',
  age: 26,
  email: 'kavya.nair@example.com',
  phone: '9876543217',
  city: 'Kochi',
  state: 'Kerala',
  membership: 'Silver',
  join_date: '2025-02-15',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: '

# Sort in Descending Order

To sort documents in **descending order**, specify **-1** as the sorting order.

The following example displays customers from the oldest to the youngest.

Equivalent SQL query:

```sql
SELECT *
FROM customers
ORDER BY age DESC;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().sort("age", -1)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b91'),
 'age': 40,
 'city': 'Chennai',
 'customer_id': 1011,
 'email': 'suresh.iyer@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-09-25',
 'membership': 'Platinum',
 'name': 'Suresh Iyer',
 'phone': '9876543220',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,
 'email': 'amit.joshi@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-10-12',
 'membership': 'Gold',
 'name': 'Amit Joshi',
 'phone': '9876543218',
 'state': 'Rajasthan'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 10

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().sort({age:-1}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b91'),
  customer_id: 1011,
  name: 'Suresh Iyer',
  gender: 'Male',
  age: 40,
  email: 'suresh.iyer@example.com',
  phone: '9876543220',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Platinum',
  join_date: '2024-09-25',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  email: 'amit.joshi@example.com',
  phone: '9876543218',
  city: 'Jaipur',
  state: 'Rajasthan',
  membership: 'Gold',
  join_date: '2024-10-12',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  ema

# Limit Query Results

The **`limit()`** method restricts the number of documents returned by a query.

This is useful when displaying only a few records, such as:

- Top 5 customers
- First 10 products
- Latest 20 orders

Equivalent SQL query:

```sql
SELECT *
FROM customers
LIMIT 5;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().limit(5)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Silver',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().limit(5).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Silver',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,

# Skip Documents

The **`skip()`** method skips a specified number of documents before returning the results.

It is commonly used with `limit()` to implement **pagination**.

Equivalent SQL concept:

```sql
OFFSET
```

The following example skips the first **3** documents.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().skip(3)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 29,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Platinum',
 'name': 'Sneha Patel',
 'phone': '9876543213',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().skip(3).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 29,
  email: 'sneha.patel@example.com',
  phone: '9876543213',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Platinum',
  join_date: '2025-03-05',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31

# Combining Multiple Operations

MongoDB allows multiple query operations to be chained together.

The following example:

1. Sorts customers by age in descending order.
2. Skips the first two customers.
3. Displays the next three customers.

Equivalent SQL query:

```sql
SELECT *
FROM customers
ORDER BY age DESC
LIMIT 3 OFFSET 2;
```

This technique is widely used in web applications for implementing pagination.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = (
    customers.find()
             .sort("age", -1)
             .skip(2)
             .limit(3)
)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Bronze',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().sort({age:-1}).skip(2).limit(3).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Bronze',
  join_date: '2025-04-01',
  is_active: true
}


# Update Documents

MongoDB provides several methods to modify existing documents.

Unlike relational databases, MongoDB updates individual fields without replacing the entire document (unless explicitly requested).

The most commonly used update methods are:

| Method | Description |
|---------|-------------|
| `update_one()` | Updates the first matching document |
| `update_many()` | Updates all matching documents |
| `replace_one()` | Replaces an existing document completely |

MongoDB update operations use **update operators** such as:

- `$set`
- `$inc`
- `$unset`

These operators modify only the required fields while leaving the remaining fields unchanged.

# Update a Single Document using `update_one()`

The **`update_one()`** method updates the **first document** that matches the specified filter.

General Syntax:

```python
collection.update_one(filter, update)
```

The following example changes the membership of customer **1002** from **Silver** to **Gold**.

Equivalent SQL query:

```sql
UPDATE customers
SET membership = 'Gold'
WHERE customer_id = 1002;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

result = customers.update_one(
    {"customer_id": 1002},
    {
        "$set": {
            "membership": "Gold"
        }
    }
)

print("Matched Documents :", result.matched_count)
print("Modified Documents:", result.modified_count)

Matched Documents : 1
Modified Documents: 1


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.updateOne(
#     {customer_id:1002},
#     {
#         $set:{
#             membership:'Gold'
#         }
#     }
# )"

# Verify the Updated Document

Let's retrieve the updated customer document to verify the changes.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"customer_id": 1002})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Gold',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({customer_id:1002}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-02-10',
  is_active: true
}


# Update Multiple Documents using `update_many()`

The **`update_many()`** method updates **all documents** that match the specified filter.

The following example upgrades every customer having **Bronze** membership to **Silver**.

Equivalent SQL query:

```sql
UPDATE customers
SET membership = 'Silver'
WHERE membership = 'Bronze';
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

result = customers.update_many(
    {
        "membership": "Bronze"
    },
    {
        "$set": {
            "membership": "Silver"
        }
    }
)

print("Matched Documents :", result.matched_count)
print("Modified Documents:", result.modified_count)

Matched Documents : 2
Modified Documents: 2


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.updateMany(
#     {membership:'Bronze'},
#     {
#         \$set:{
#             membership:'Silver'
#         }
#     }
# )"

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": "Silver"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b8b'),
 'age': 35,
 'city': 'Chennai',
 'customer_id': 1005,
 'email': 'vikram.singh@example.com',
 'gender': 'Male',
 'is_active': False,
 'join_date': '2024-11-18',
 'membership': 'Silver',
 'name': 'Vikram Singh',
 'phone': '9876543214',
 'state': 'Tamil Nadu'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Silver',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8e'),
 'age': 26,
 'city': 'Kochi',
 'customer_id': 1008,
 'email': 'kavya.nair@example.com',
 'gender': 'Female',
 'is_active': False,
 'join_date': '2025-02-15',
 'membership': 'Silver',
 'name': 'Kavya Nair',
 'phone': '9876543217',
 'state': 'Kerala'}
{'_id': ObjectId('6a5cba4483106d9367c89b90'),
 'age': 24,
 'city': 'Delhi',
 'customer_id': 1010,
 'e

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:'Silver'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b8b'),
  customer_id: 1005,
  name: 'Vikram Singh',
  gender: 'Male',
  age: 35,
  email: 'vikram.singh@example.com',
  phone: '9876543214',
  city: 'Chennai',
  state: 'Tamil Nadu',
  membership: 'Silver',
  join_date: '2024-11-18',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Silver',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8e'),
  customer_id: 1008,
  name: 'Kavya Nair',
  gender: 'Female',
  age: 26,
  email: 'kavya.nair@example.com',
  phone: '9876543217',
  city: 'Kochi',
  state: 'Kerala',
  membership: 'Silver',
  join_date: '2025-02-15',
  is_active: false
}
{
  _id: ObjectId('6a5cba4483106d9367c89b90'),
  customer_id: 1010,
  name: 'Neha Sharma',
  gender: 'Female',
  age: 24,
  emai

# Increment a Numeric Field using `$inc`

The **`$inc`** operator increases or decreases the value of a numeric field.

This operator is useful for:

- Increasing product stock
- Updating customer reward points
- Incrementing counters
- Recording page visits

The following example increases the age of customer **1001** by **1**.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

result = customers.update_one(
    {"customer_id": 1001},
    {
        "$inc": {
            "age": 1
        }
    }
)

print("Modified Documents:", result.modified_count)

Modified Documents: 1


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.updateOne(
#     {customer_id:1001},
#     {
#         \$inc:{
#             age:1
#         }
#     }
# )"

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"customer_id":1001})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({customer_id:1001}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 29,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}


# Remove a Field using `$unset`

The **`$unset`** operator removes one or more fields from a document.

Unlike `$set`, which adds or modifies a field, `$unset` permanently deletes the specified field.

General Syntax:

```python
collection.update_one(
    filter,
    {
        "$unset": {
            field_name: ""
        }
    }
)
```

The value associated with the field inside `$unset` is ignored. An empty string (`""`) is commonly used.

The following example removes the **phone** field from the customer whose `customer_id` is **1003**.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

result = customers.update_one(
    {"customer_id": 1003},
    {
        "$unset": {
            "phone": ""
        }
    }
)

print("Matched Documents :", result.matched_count)
print("Modified Documents:", result.modified_count)

Matched Documents : 1
Modified Documents: 1


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.updateOne(
#     {customer_id:1003},
#     {
#         \$unset:{
#             phone:''
#         }
#     }
# )"

# Verify the Updated Document

Let's verify that the **phone** field has been removed.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"customer_id": 1003})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({customer_id:1003}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}


# Replace a Document using `replace_one()`

The **`replace_one()`** method completely replaces an existing document with a new document.

Unlike `update_one()`, which modifies only selected fields, `replace_one()` removes all existing fields (except `_id`) and replaces them with the new document.

General Syntax:

```python
collection.replace_one(filter, replacement_document)
```

> **Important:**  
> The replacement document should contain all the fields that you want to preserve. Any fields that are omitted will be removed from the document.

The following example replaces the customer with `customer_id = 1004` using a new document.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

new_customer = {
    "customer_id": 1004,
    "name": "Sneha Patel",
    "gender": "Female",
    "age": 30,
    "email": "sneha.patel@example.com",
    "city": "Mumbai",
    "state": "Maharashtra",
    "membership": "Diamond",
    "join_date": "2025-03-05",
    "is_active": True
}

result = customers.replace_one(
    {"customer_id": 1004},
    new_customer
)

print("Matched Documents :", result.matched_count)
print("Modified Documents:", result.modified_count)

Matched Documents : 1
Modified Documents: 1


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.replaceOne(
#     {customer_id:1004},
#     {
#         customer_id:1004,
#         name:'Sneha Patel',
#         gender:'Female',
#         age:30,
#         email:'sneha.patel@example.com',
#         city:'Mumbai',
#         state:'Maharashtra',
#         membership:'Diamond',
#         join_date:'2025-03-05',
#         is_active:true
#     }
# )"

# Verify the Replaced Document

Retrieve the document again to verify that it has been completely replaced.

Observe that the **phone** field is no longer present because it was not included in the replacement document.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"customer_id": 1004})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 30,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-03-05',
 'membership': 'Diamond',
 'name': 'Sneha Patel',
 'state': 'Maharashtra'}


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({customer_id:1004}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 30,
  email: 'sneha.patel@example.com',
  city: 'Mumbai',
  state: 'Maharashtra',
  membership: 'Diamond',
  join_date: '2025-03-05',
  is_active: true
}


# Delete Documents

MongoDB provides methods to remove documents from a collection.

The two primary methods are:

| Method | Description |
|---------|-------------|
| `delete_one()` | Deletes the first document that matches the filter |
| `delete_many()` | Deletes all documents that match the filter |

Deleting documents permanently removes them from the collection.

# Delete a Single Document using `delete_one()`

The **`delete_one()`** method removes the first document that matches the specified filter.

General Syntax:

```python
collection.delete_one(filter)
```

The following example deletes the customer whose `customer_id` is **1011**.

Equivalent SQL query:

```sql
DELETE FROM customers
WHERE customer_id = 1011;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

result = customers.delete_one(
    {"customer_id": 1011}
)

print("Deleted Documents:", result.deleted_count)

Deleted Documents: 1


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.deleteOne(
#     {customer_id:1011}
# )"

# Verify the Deletion

Retrieve the deleted document.

Since the document no longer exists, no document should be returned.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

customer = customers.find_one({"customer_id": 1011})

pprint(customer)

None


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({customer_id:1011}).forEach(doc => printjson(doc))"

# Delete Multiple Documents using `delete_many()`

The **`delete_many()`** method removes every document that matches the specified filter.

The following example deletes all **inactive customers**.

Equivalent SQL query:

```sql
DELETE FROM customers
WHERE is_active = FALSE;
```

In [ ]:
# =====================================
# PyMongo Command
# =====================================

result = customers.delete_many(
    {
        "is_active": False
    }
)

print("Deleted Documents:", result.deleted_count)

Deleted Documents: 2


In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.deleteMany(
#     {is_active:false}
# )"

# Verify the Remaining Documents

Let's retrieve all remaining customers after the deletion.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find()

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Gold',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 30,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.pate

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find().forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 29,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 30,
  email: 'sneha.patel@ex

# Delete a Collection using `drop()`

The **`drop()`** method permanently removes an entire collection, including all its documents and indexes.

General Syntax:

```python
collection.drop()
```

> **Warning:**  
> This operation cannot be undone.

For demonstration purposes, we will create a temporary collection and then delete it.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

temp = db["temp_collection"]

temp.insert_one({"message": "Temporary Document"})

db.list_collection_names()

['temp_collection', 'customers']

In [ ]:
# =====================================
# PyMongo Command
# =====================================

temp.drop()

db.list_collection_names()

['customers']

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db = db.getSiblingDB('ecommerce');
# db.temp_collection.insertOne({message:'Temporary Document'});
# db.temp_collection.drop();
# "

# Delete a Database using `drop_database()`

MongoDB allows an entire database to be deleted using the **`drop_database()`** method.

General Syntax:

```python
client.drop_database(database_name)
```

> **Warning:**  
> Dropping a database permanently removes all collections, documents, indexes, and metadata contained within it.

In this notebook, we will **not execute** this command because the **ecommerce** database will be required in the Aggregation Pipeline section.

The following commands are shown only for reference.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

# client.drop_database("ecommerce")

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "db.getSiblingDB('ecommerce').dropDatabase()"

# Aggregation Pipeline

In real-world applications, we often need more than simply retrieving documents.

For example, we may want to answer questions such as:

- How many customers belong to each membership category?
- What is the average age of customers?
- Which city has the highest number of customers?
- How many active customers are there?

These types of questions require **data analysis** rather than simple document retrieval.

MongoDB provides the **Aggregation Pipeline** to perform these operations efficiently.

The Aggregation Pipeline processes documents through a sequence of stages. Each stage performs a specific operation on the data and passes the result to the next stage.

General Syntax:

```python
collection.aggregate([
    Stage 1,
    Stage 2,
    Stage 3,
    ...
])
```

Each stage transforms the data before passing it to the next stage, similar to how water flows through a series of pipes, where each pipe performs a specific task.

This step-by-step processing makes the Aggregation Pipeline powerful for reporting, analytics, and business intelligence.

# Aggregation Pipeline vs `find()`

Both **`find()`** and **`aggregate()`** are used to retrieve data from MongoDB, but they serve different purposes.

| `find()` | `aggregate()` |
|----------|---------------|
| Retrieves documents | Processes and analyzes documents |
| Simple filtering | Complex data processing |
| Returns original documents | Returns transformed or summarized results |
| Used for CRUD operations | Used for reporting and analytics |

Use **`find()`** when you want to retrieve documents.

Use **`aggregate()`** when you want to summarize, calculate, group, or transform data.

# Pipeline Stages

An Aggregation Pipeline consists of one or more **stages**.

Each stage performs a specific operation on the incoming documents.

Some commonly used stages are:

| Stage | Purpose |
|--------|---------|
| `$match` | Filters documents |
| `$project` | Selects or computes fields |
| `$group` | Groups documents and performs calculations |
| `$sort` | Sorts documents |
| `$limit` | Limits the number of documents |
| `$count` | Counts documents |
| `$unwind` | Splits array elements into separate documents |
| `$lookup` | Joins documents from another collection |

The output of one stage becomes the input of the next stage.

# Stage 1: `$match`

The **`$match`** stage filters documents based on specified conditions.

Its purpose is similar to the **`find()`** method.

Equivalent SQL query:

```sql
SELECT *
FROM customers
WHERE membership = 'Gold';
```

The following example retrieves all customers whose membership is **Gold**.

In [ ]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$match": {
            "membership": "Gold"
        }
    }
]

cursor = customers.aggregate(pipeline)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Gold',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.da

In [ ]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{$match:{membership:'Gold'}}]).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 29,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@exam

# Why use `$match` instead of `find()`?

You may wonder why we use `$match` when `find()` already performs filtering.

The answer is that `$match` is designed to work **inside an Aggregation Pipeline**.

For example, we can first filter customers using `$match`, then group them using `$group`, sort the results using `$sort`, and finally limit the output using `$limit`.

This makes the Aggregation Pipeline much more powerful than a simple `find()` query.

# Stage 2: `$group`

The **`$group`** stage groups documents based on a common field and performs calculations on each group.

It is similar to the **`GROUP BY`** clause in SQL.

General Syntax:

```python
collection.aggregate([
    {
        "$group": {
            "_id": "<grouping_field>",
            "<new_field>": {
                "<aggregation_operator>": "<expression>"
            }
        }
    }
])
```

Equivalent SQL query:

```sql
SELECT grouping_column,
       AGGREGATE_FUNCTION(column)
FROM table_name
GROUP BY grouping_column;
```

The `_id` field specifies the field used for grouping.

The remaining fields contain the calculated values for each group.

# Count Customers by Membership

The following example groups customers based on their **membership** and counts the number of customers in each membership category.

Equivalent SQL query:

```sql
SELECT membership,
       COUNT(*)
FROM customers
GROUP BY membership;
```

In [103]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$group": {
            "_id": "$membership",
            "total_customers": {
                "$sum": 1
            }
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': 'Diamond', 'total_customers': 1}
{'_id': 'Gold', 'total_customers': 5}
{'_id': 'Silver', 'total_customers': 2}


In [104]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $group: { _id: '$membership', total_customers: { $sum: 1 } } }]).forEach(doc => printjson(doc))"

{
  _id: 'Diamond',
  total_customers: 1
}
{
  _id: 'Gold',
  total_customers: 5
}
{
  _id: 'Silver',
  total_customers: 2
}


# Understanding the `$sum` Operator

The **`$sum`** operator performs addition during aggregation.

When used as:

```javascript
{
    "$sum": 1
}
```

MongoDB adds **1** for every document in the group.

Therefore,

```text
Gold
Gold
Gold
```

becomes

```text
Gold → 3
```

This is equivalent to the SQL function:

```sql
COUNT(*)
```

# Count Customers by City

We can group documents using any field.

The following example counts the number of customers in each city.

Equivalent SQL query:

```sql
SELECT city,
       COUNT(*)
FROM customers
GROUP BY city;
```

In [105]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$group": {
            "_id": "$city",
            "total_customers": {
                "$sum": 1
            }
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': 'Pune', 'total_customers': 1}
{'_id': 'Mumbai', 'total_customers': 1}
{'_id': 'Hyderabad', 'total_customers': 2}
{'_id': 'Bengaluru', 'total_customers': 1}
{'_id': 'Jaipur', 'total_customers': 1}
{'_id': 'Delhi', 'total_customers': 1}
{'_id': 'Kolkata', 'total_customers': 1}


In [106]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $group: { _id: '$city', total_customers: { $sum: 1 } } }]).forEach(doc => printjson(doc))"

{
  _id: 'Delhi',
  total_customers: 1
}
{
  _id: 'Bengaluru',
  total_customers: 1
}
{
  _id: 'Hyderabad',
  total_customers: 2
}
{
  _id: 'Mumbai',
  total_customers: 1
}
{
  _id: 'Pune',
  total_customers: 1
}
{
  _id: 'Jaipur',
  total_customers: 1
}
{
  _id: 'Kolkata',
  total_customers: 1
}


# Calculate Average Age using `$avg`

The **`$avg`** operator calculates the average value of a numeric field.

The following example calculates the average age of all customers.

Equivalent SQL query:

```sql
SELECT AVG(age)
FROM customers;
```

In [107]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$group": {
            "_id": None,
            "average_age": {
                "$avg": "$age"
            }
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': None, 'average_age': 29.5}


In [108]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $group: { _id: null, average_age: { $avg: '$age' } } }]).forEach(doc => printjson(doc))"

{
  _id: null,
  average_age: 29.5
}


# Find Minimum and Maximum Age

MongoDB provides the **`$min`** and **`$max`** operators to determine the smallest and largest values in a group.

Equivalent SQL query:

```sql
SELECT MIN(age),
       MAX(age)
FROM customers;
```

In [109]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$group": {
            "_id": None,
            "minimum_age": {
                "$min": "$age"
            },
            "maximum_age": {
                "$max": "$age"
            }
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': None, 'maximum_age': 38, 'minimum_age': 24}


In [110]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $group: { _id: null, minimum_age: { $min: '$age' }, maximum_age: { $max: '$age' } } }]).forEach(doc => printjson(doc))"

{
  _id: null,
  minimum_age: 24,
  maximum_age: 38
}


# Stage 3: `$project`

The **`$project`** stage reshapes the documents that pass through the aggregation pipeline.

It is similar to **Projection** in the `find()` method, but it is much more powerful because it can:

- Include fields
- Exclude fields
- Rename fields
- Compute new fields

General Syntax:

```python
collection.aggregate([
    {
        "$project": {
            field1: 1,
            field2: 1,
            "_id": 0
        }
    }
])
```

The following example displays only the customer's **name**, **city**, and **membership**.

In [111]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "city": 1,
            "membership": 1
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'city': 'Hyderabad', 'membership': 'Gold', 'name': 'Rahul Sharma'}
{'city': 'Hyderabad', 'membership': 'Gold', 'name': 'Priya Reddy'}
{'city': 'Bengaluru', 'membership': 'Gold', 'name': 'Arjun Kumar'}
{'city': 'Mumbai', 'membership': 'Diamond', 'name': 'Sneha Patel'}
{'city': 'Kolkata', 'membership': 'Gold', 'name': 'Ananya Das'}
{'city': 'Pune', 'membership': 'Silver', 'name': 'Rohit Verma'}
{'city': 'Jaipur', 'membership': 'Gold', 'name': 'Amit Joshi'}
{'city': 'Delhi', 'membership': 'Silver', 'name': 'Neha Sharma'}


In [112]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $project: { _id: 0, name: 1, city: 1, membership: 1 } }]).forEach(doc => printjson(doc))"

{
  name: 'Rahul Sharma',
  city: 'Hyderabad',
  membership: 'Gold'
}
{
  name: 'Priya Reddy',
  city: 'Hyderabad',
  membership: 'Gold'
}
{
  name: 'Arjun Kumar',
  city: 'Bengaluru',
  membership: 'Gold'
}
{
  name: 'Sneha Patel',
  city: 'Mumbai',
  membership: 'Diamond'
}
{
  name: 'Ananya Das',
  city: 'Kolkata',
  membership: 'Gold'
}
{
  name: 'Rohit Verma',
  city: 'Pune',
  membership: 'Silver'
}
{
  name: 'Amit Joshi',
  city: 'Jaipur',
  membership: 'Gold'
}
{
  name: 'Neha Sharma',
  city: 'Delhi',
  membership: 'Silver'
}


# Stage 4: `$sort`

The **`$sort`** stage sorts the documents produced by the previous pipeline stage.

General Syntax:

```python
{
    "$sort": {
        field: 1
    }
}
```

where:

- **1** → Ascending Order
- **-1** → Descending Order

The following example sorts customers by **age** in descending order.

In [113]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$sort": {
            "age": -1
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,
 'email': 'amit.joshi@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-10-12',
 'membership': 'Gold',
 'name': 'Amit Joshi',
 'phone': '9876543218',
 'state': 'Rajasthan'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8d'),
 'age': 31,
 'city': 'Pune',
 'customer_id': 1007,
 'email': 'rohit.verma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-04-01',
 'membership': 'Silver',
 'name': 'Rohit Verma',
 'phone': '9876543216',
 'state': 'Maharashtra'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 30,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.patel@example.

In [114]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $sort: { age: -1 } }]).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  email: 'amit.joshi@example.com',
  phone: '9876543218',
  city: 'Jaipur',
  state: 'Rajasthan',
  membership: 'Gold',
  join_date: '2024-10-12',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8d'),
  customer_id: 1007,
  name: 'Rohit Verma',
  gender: 'Male',
  age: 31,
  email: 'rohit.verma@example.com',
  phone: '9876543216',
  city: 'Pune',
  state: 'Maharashtra',
  membership: 'Silver',
  join_date: '2025-04-01',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8a'),
  customer_id: 1004,
  name: 'Sneha Patel',
  gender: 'Female',
  age: 30,
  email: 'sneha.patel@example.com'

# Stage 5: `$limit`

The **`$limit`** stage restricts the number of documents returned by the aggregation pipeline.

It is commonly used to retrieve the **Top N** results.

The following example returns the first **3** customers.

In [115]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$limit": 3
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Gold',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}


In [116]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $limit: 3 }]).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 29,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-02-10',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}


# Stage 6: `$count`

The **`$count`** stage counts the number of documents that reach this stage of the pipeline.

Unlike `count_documents()`, `$count` is used **inside an aggregation pipeline**.

The following example counts the total number of customer documents.

In [117]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$count": "total_customers"
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'total_customers': 8}


In [118]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $count: 'total_customers' }]).forEach(doc => printjson(doc))"

{
  total_customers: 8
}


# Combining Multiple Aggregation Stages

The true power of the Aggregation Pipeline comes from combining multiple stages.

The following example performs the following operations:

1. Filters **active customers**.
2. Groups them by **membership**.
3. Counts the number of customers in each membership.
4. Sorts the results in descending order of customer count.

Equivalent SQL query:

```sql
SELECT membership,
       COUNT(*) AS total_customers
FROM customers
WHERE is_active = TRUE
GROUP BY membership
ORDER BY total_customers DESC;
```

In [119]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$match": {
            "is_active": True
        }
    },
    {
        "$group": {
            "_id": "$membership",
            "total_customers": {
                "$sum": 1
            }
        }
    },
    {
        "$sort": {
            "total_customers": -1
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': 'Gold', 'total_customers': 5}
{'_id': 'Silver', 'total_customers': 2}
{'_id': 'Diamond', 'total_customers': 1}


In [120]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $match: { is_active: true } }, { $group: { _id: '$membership', total_customers: { $sum: 1 } } }, { $sort: { total_customers: -1 } }]).forEach(doc => printjson(doc))"

{
  _id: 'Gold',
  total_customers: 5
}
{
  _id: 'Silver',
  total_customers: 2
}
{
  _id: 'Diamond',
  total_customers: 1
}


# Create a Demo Collection

To understand the **`$lookup`** stage, we need two related collections.

We already have the **customers** collection.

Now, we will create an **orders** collection.

Relationship:

- One customer can place multiple orders.
- Each order belongs to exactly one customer.

This represents a **One-to-Many Relationship**, similar to foreign key relationships in relational databases.

In [121]:
# =====================================
# PyMongo Command
# =====================================

orders = db["orders"]

orders.drop()

orders.insert_many([
    {
        "order_id": 101,
        "customer_id": 1001,
        "product": "Laptop",
        "quantity": 1,
        "price": 65000,
        "order_date": "2025-06-10"
    },
    {
        "order_id": 102,
        "customer_id": 1001,
        "product": "Wireless Mouse",
        "quantity": 2,
        "price": 1200,
        "order_date": "2025-06-12"
    },
    {
        "order_id": 103,
        "customer_id": 1003,
        "product": "Mechanical Keyboard",
        "quantity": 1,
        "price": 4500,
        "order_date": "2025-06-15"
    },
    {
        "order_id": 104,
        "customer_id": 1005,
        "product": "Monitor",
        "quantity": 1,
        "price": 18000,
        "order_date": "2025-06-18"
    },
    {
        "order_id": 105,
        "customer_id": 1005,
        "product": "USB Hub",
        "quantity": 1,
        "price": 900,
        "order_date": "2025-06-20"
    }
])

print("Orders collection created successfully.")

Orders collection created successfully.


In [122]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "db.getSiblingDB('ecommerce').orders.insertMany([...])"

# Relationship Between Collections

The **customers** and **orders** collections are related using the **customer_id** field.

```mermaid
erDiagram

CUSTOMERS ||--o{ ORDERS : places

CUSTOMERS {
    int customer_id
    string name
    string city
}

ORDERS {
    int order_id
    int customer_id
    string product
    int quantity
    double price
}
```

Each customer can have multiple orders.

The `customer_id` field acts like a **foreign key** in a relational database.

# Stage 7: `$lookup`

The **`$lookup`** stage joins documents from two collections.

It is equivalent to the **JOIN** operation in relational databases.

General Syntax:

```python
{
    "$lookup": {
        "from": "<collection_name>",
        "localField": "<field_in_current_collection>",
        "foreignField": "<field_in_other_collection>",
        "as": "<output_array>"
    }
}
```

Equivalent SQL query:

```sql
SELECT *
FROM customers
JOIN orders
ON customers.customer_id = orders.customer_id;
```

Unlike SQL, MongoDB stores the matching documents inside an **array**.

In [123]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$lookup": {
            "from": "orders",
            "localField": "customer_id",
            "foreignField": "customer_id",
            "as": "orders"
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'orders': [{'_id': ObjectId('6a5cdb7cf008f20b65742fc5'),
             'customer_id': 1001,
             'order_date': '2025-06-10',
             'order_id': 101,
             'price': 65000,
             'product': 'Laptop',
             'quantity': 1},
            {'_id': ObjectId('6a5cdb7cf008f20b65742fc6'),
             'customer_id': 1001,
             'order_date': '2025-06-12',
             'order_id': 102,
             'price': 1200,
             'product': 'Wireless Mouse',
             'quantity': 2}],
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,


In [124]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $lookup: { from: 'orders', localField: 'customer_id', foreignField: 'customer_id', as: 'orders' } }]).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 29,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true,
  orders: [
    {
      _id: ObjectId('6a5cdb7cf008f20b65742fc5'),
      order_id: 101,
      customer_id: 1001,
      product: 'Laptop',
      quantity: 1,
      price: 65000,
      order_date: '2025-06-10'
    },
    {
      _id: ObjectId('6a5cdb7cf008f20b65742fc6'),
      order_id: 102,
      customer_id: 1001,
      product: 'Wireless Mouse',
      quantity: 2,
      price: 1200,
      order_date: '2025-06-12'
    }
  ]
}
{
  _id: ObjectId('6a5cba4483106d9367c89b88'),
  customer_id: 1002,
  name: 'Priya Reddy',
  gender: 'Female',
  age: 25,
  email: 'priya.reddy@example.com',
  phone: '9876543211',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-02-10',
  is_

# Understanding the Result

Notice that the **orders** are not merged directly into the customer document.

Instead, MongoDB creates a new field named **orders**.

This field contains an **array** of all matching order documents.

If a customer has not placed any orders, the **orders** array will be empty.

This behavior is different from SQL joins and allows MongoDB to naturally represent one-to-many relationships.

# Stage 8: `$unwind`

The **`$unwind`** stage deconstructs an **array field** into multiple documents.

Each element of the array becomes a separate document.

General Syntax:

```python
{
    "$unwind": "$array_field"
}
```

The `$unwind` stage is commonly used after **`$lookup`**, because `$lookup` stores the matching documents inside an array.

By unwinding the array, each array element can be processed individually.

# Why Do We Need `$unwind`?

Consider the following document produced by the `$lookup` stage.

```json
{
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "orders": [
        {
            "order_id": 101,
            "product": "Laptop"
        },
        {
            "order_id": 102,
            "product": "Wireless Mouse"
        }
    ]
}
```

Although the customer has two orders, they are stored inside a single array.

The `$unwind` stage converts this into two separate documents.

Document 1

```json
{
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "orders": {
        "order_id": 101,
        "product": "Laptop"
    }
}
```

Document 2

```json
{
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "orders": {
        "order_id": 102,
        "product": "Wireless Mouse"
    }
}
```

This makes it easier to perform filtering, grouping, sorting, and reporting on individual order records.

In [125]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$lookup": {
            "from": "orders",
            "localField": "customer_id",
            "foreignField": "customer_id",
            "as": "orders"
        }
    },
    {
        "$unwind": "$orders"
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'orders': {'_id': ObjectId('6a5cdb7cf008f20b65742fc5'),
            'customer_id': 1001,
            'order_date': '2025-06-10',
            'order_id': 101,
            'price': 65000,
            'product': 'Laptop',
            'quantity': 1},
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'orders': {'_id': ObjectId('6a5cdb7cf008f20b65742fc6'),
            'customer_id': 1001,
            'order_date': '2025-06-12',
            'order_id': 102,
            'price': 1200,
         

In [126]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $lookup: { from: 'orders', localField: 'customer_id', foreignField: 'customer_id', as: 'orders' } }, { $unwind: '$orders' }]).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 29,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true,
  orders: {
    _id: ObjectId('6a5cdb7cf008f20b65742fc5'),
    order_id: 101,
    customer_id: 1001,
    product: 'Laptop',
    quantity: 1,
    price: 65000,
    order_date: '2025-06-10'
  }
}
{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 29,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true,
  orders: {
    _id: ObjectId('6a5cdb7cf008f20b65742fc6'),
    order_id: 102,
    customer_id: 1001,
    product: 'Wireless Mouse',
    quantity: 2,
    price: 1200,
    order_date: '2025-06-12'
  }
}
{
  _id: ObjectId('6a

# Combining `$lookup`, `$unwind`, and `$project`

In real-world applications, multiple aggregation stages are often combined.

The following pipeline:

1. Joins the **customers** and **orders** collections.
2. Converts the orders array into individual documents.
3. Displays only the required fields.

This produces a report where each row represents one customer order.

Equivalent SQL query:

```sql
SELECT
    c.name,
    c.city,
    o.product,
    o.quantity,
    o.price
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id;
```

In [127]:
# =====================================
# PyMongo Command
# =====================================

pipeline = [
    {
        "$lookup": {
            "from": "orders",
            "localField": "customer_id",
            "foreignField": "customer_id",
            "as": "orders"
        }
    },
    {
        "$unwind": "$orders"
    },
    {
        "$project": {
            "_id": 0,
            "Customer Name": "$name",
            "City": "$city",
            "Product": "$orders.product",
            "Quantity": "$orders.quantity",
            "Price": "$orders.price"
        }
    }
]

cursor = customers.aggregate(pipeline)

for document in cursor:
    pprint(document)

{'City': 'Hyderabad',
 'Customer Name': 'Rahul Sharma',
 'Price': 65000,
 'Product': 'Laptop',
 'Quantity': 1}
{'City': 'Hyderabad',
 'Customer Name': 'Rahul Sharma',
 'Price': 1200,
 'Product': 'Wireless Mouse',
 'Quantity': 2}
{'City': 'Bengaluru',
 'Customer Name': 'Arjun Kumar',
 'Price': 4500,
 'Product': 'Mechanical Keyboard',
 'Quantity': 1}


In [128]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.aggregate([{ $lookup: { from: 'orders', localField: 'customer_id', foreignField: 'customer_id', as: 'orders' } }, { $unwind: '$orders' }, { $project: { _id: 0, 'Customer Name': '$name', City: '$city', Product: '$orders.product', Quantity: '$orders.quantity', Price: '$orders.price' } }]).forEach(doc => printjson(doc))"

{
  'Customer Name': 'Rahul Sharma',
  City: 'Hyderabad',
  Product: 'Laptop',
  Quantity: 1,
  Price: 65000
}
{
  'Customer Name': 'Rahul Sharma',
  City: 'Hyderabad',
  Product: 'Wireless Mouse',
  Quantity: 2,
  Price: 1200
}
{
  'Customer Name': 'Arjun Kumar',
  City: 'Bengaluru',
  Product: 'Mechanical Keyboard',
  Quantity: 1,
  Price: 4500
}


# Indexing

As the amount of data in a collection grows, searching every document becomes slower.

By default, MongoDB scans every document in a collection to find matching records. This process is called a **Collection Scan (COLLSCAN)**.

An **Index** is a special data structure that stores the values of one or more fields in sorted order. Instead of scanning every document, MongoDB searches the index to quickly locate the required documents.

Advantages of Indexing:

- Improves query performance.
- Reduces query execution time.
- Speeds up sorting operations.
- Essential for large collections.

> **Note:** Creating too many indexes consumes additional storage and slows down insert, update, and delete operations because every index must also be updated.

# Types of Indexes

MongoDB supports several types of indexes.

| Index Type | Description |
|------------|-------------|
| Single Field Index | Created on one field |
| Compound Index | Created on multiple fields |
| Unique Index | Prevents duplicate values |
| Text Index | Supports full-text search |
| Multikey Index | Indexes array fields |
| Hashed Index | Used for hashed sharding |
| Geospatial Index | Used for location-based queries |

In this notebook, we will focus on the most commonly used index types:

- Single Field Index
- Compound Index
- Unique Index
- Text Index

# Create a Single Field Index

A **Single Field Index** is created on one field.

The following example creates an index on the **city** field.

General Syntax:

```python
collection.create_index(field_name)
```

In [129]:
# =====================================
# PyMongo Command
# =====================================

customers.create_index("city")

'city_1'

In [130]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.createIndex({city:1})"

city_1


# List Indexes

The **`list_indexes()`** method displays all indexes available in a collection.

Notice that MongoDB automatically creates an index on the **`_id`** field when the collection is created.

In [131]:
# =====================================
# PyMongo Command
# =====================================

for index in customers.list_indexes():
    pprint(index)

SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('city', 1)])), ('name', 'city_1')])


In [132]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.getIndexes()"

[
  { v: 2, key: { _id: 1 }, name: '_id_' },
  { v: 2, key: { city: 1 }, name: 'city_1' }
]


# Create a Compound Index

A **Compound Index** is created on multiple fields.

It is useful when queries frequently filter using more than one field.

The following example creates a compound index on **city** and **membership**.

In [133]:
# =====================================
# PyMongo Command
# =====================================

customers.create_index([
    ("city", 1),
    ("membership", 1)
])

'city_1_membership_1'

In [134]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.createIndex({city:1,membership:1})"

city_1_membership_1


# Create a Unique Index

A **Unique Index** ensures that no two documents contain the same value for the indexed field.

If a duplicate value is inserted, MongoDB raises an error.

The following example creates a unique index on the **email** field.

In [135]:
# =====================================
# PyMongo Command
# =====================================

customers.create_index(
    "email",
    unique=True
)

'email_1'

In [136]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.createIndex({email:1},{unique:true})"

email_1


# Drop an Index

Indexes that are no longer required can be removed using the **`drop_index()`** method.

The index name can be obtained using **`list_indexes()`**.

In [137]:
# =====================================
# PyMongo Command
# =====================================

customers.drop_index("city_1")

In [138]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.dropIndex('city_1')"

{ nIndexesWas: 3, ok: 1 }


In [139]:
# =====================================
# PyMongo Command
# =====================================

for index in customers.list_indexes():
    pprint(index)

SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('city', 1), ('membership', 1)])), ('name', 'city_1_membership_1')])
SON([('v', 2), ('key', SON([('email', 1)])), ('name', 'email_1'), ('unique', True)])


In [140]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.getIndexes()"

[
  { v: 2, key: { _id: 1 }, name: '_id_' },
  {
    v: 2,
    key: { city: 1, membership: 1 },
    name: 'city_1_membership_1'
  },
  { v: 2, key: { email: 1 }, name: 'email_1', unique: true }
]


# Text Search

Searching for text is a common requirement in many applications.

Examples include:

- Searching products
- Searching books
- Searching blogs
- Searching news articles

Although the **`$regex`** operator can search text, it becomes slow for large collections because MongoDB scans many documents.

MongoDB provides **Text Indexes** and the **`$text`** operator to perform efficient full-text searches.

> **Important:**  
> Before using the **`$text`** operator, a **Text Index** must be created on one or more string fields.

# Create a Demo Collection

The following collection contains sample products with descriptions.

This collection will be used to demonstrate **Text Search**.

In [141]:
# =====================================
# PyMongo Command
# =====================================

products = db["products"]

products.drop()

products.insert_many([
    {
        "product_id": 1,
        "name": "Gaming Laptop",
        "description": "High performance gaming laptop with NVIDIA graphics card."
    },
    {
        "product_id": 2,
        "name": "Wireless Mouse",
        "description": "Ergonomic wireless mouse with rechargeable battery."
    },
    {
        "product_id": 3,
        "name": "Mechanical Keyboard",
        "description": "RGB mechanical keyboard designed for gaming and programming."
    },
    {
        "product_id": 4,
        "name": "USB Hub",
        "description": "USB Type-C hub with HDMI and Ethernet ports."
    }
])

print("Products collection created successfully.")

Products collection created successfully.


In [142]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "db.getSiblingDB('ecommerce').products.insertMany([...])"

# Create a Text Index

A **Text Index** allows MongoDB to perform efficient full-text searches.

The following example creates a Text Index on the **description** field.

In [143]:
# =====================================
# PyMongo Command
# =====================================

products.create_index([
    ("description", "text")
])

'description_text'

In [144]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').products.createIndex({description:'text'})"

description_text


# Search Using the `$text` Operator

The **`$text`** operator searches documents using the Text Index.

The following example searches for the word **gaming**.

In [145]:
# =====================================
# PyMongo Command
# =====================================

cursor = products.find(
    {
        "$text": {
            "$search": "gaming"
        }
    }
)

for product in cursor:
    pprint(product)

{'_id': ObjectId('6a5d2c95f008f20b65742fcc'),
 'description': 'RGB mechanical keyboard designed for gaming and programming.',
 'name': 'Mechanical Keyboard',
 'product_id': 3}
{'_id': ObjectId('6a5d2c95f008f20b65742fca'),
 'description': 'High performance gaming laptop with NVIDIA graphics card.',
 'name': 'Gaming Laptop',
 'product_id': 1}


In [146]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').products.find({$text:{$search:'gaming'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5d2c95f008f20b65742fcc'),
  product_id: 3,
  name: 'Mechanical Keyboard',
  description: 'RGB mechanical keyboard designed for gaming and programming.'
}
{
  _id: ObjectId('6a5d2c95f008f20b65742fca'),
  product_id: 1,
  name: 'Gaming Laptop',
  description: 'High performance gaming laptop with NVIDIA graphics card.'
}


# Search Multiple Words

The `$search` operator accepts multiple search terms.

MongoDB returns documents that match one or more of the specified words.

The following example searches for:

- gaming
- wireless

In [147]:
# =====================================
# PyMongo Command
# =====================================

cursor = products.find(
    {
        "$text": {
            "$search": "gaming wireless"
        }
    }
)

for product in cursor:
    pprint(product)

{'_id': ObjectId('6a5d2c95f008f20b65742fcc'),
 'description': 'RGB mechanical keyboard designed for gaming and programming.',
 'name': 'Mechanical Keyboard',
 'product_id': 3}
{'_id': ObjectId('6a5d2c95f008f20b65742fca'),
 'description': 'High performance gaming laptop with NVIDIA graphics card.',
 'name': 'Gaming Laptop',
 'product_id': 1}
{'_id': ObjectId('6a5d2c95f008f20b65742fcb'),
 'description': 'Ergonomic wireless mouse with rechargeable battery.',
 'name': 'Wireless Mouse',
 'product_id': 2}


In [148]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').products.find({$text:{$search:'gaming wireless'}}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5d2c95f008f20b65742fcc'),
  product_id: 3,
  name: 'Mechanical Keyboard',
  description: 'RGB mechanical keyboard designed for gaming and programming.'
}
{
  _id: ObjectId('6a5d2c95f008f20b65742fca'),
  product_id: 1,
  name: 'Gaming Laptop',
  description: 'High performance gaming laptop with NVIDIA graphics card.'
}
{
  _id: ObjectId('6a5d2c95f008f20b65742fcb'),
  product_id: 2,
  name: 'Wireless Mouse',
  description: 'Ergonomic wireless mouse with rechargeable battery.'
}


# `$regex` vs `$text`

Both `$regex` and `$text` can search text, but they serve different purposes.

| `$regex` | `$text` |
|----------|----------|
| Pattern matching | Full-text search |
| Does not require a Text Index | Requires a Text Index |
| Slower on large collections | Faster on indexed text |
| Matches characters and patterns | Matches words and phrases |
| Suitable for small datasets | Suitable for production systems |

Use **`$regex`** when matching patterns such as email addresses or names.

Use **`$text`** when searching documents based on keywords.

# Bulk Write Operations

So far, we have performed one operation at a time:

- `insert_one()`
- `update_one()`
- `delete_one()`
- `replace_one()`

In real-world applications, it is common to perform multiple operations together.

Instead of sending each operation separately, MongoDB allows multiple operations to be executed in a **single request** using the **`bulk_write()`** method.

Advantages of Bulk Write:

- Reduces network communication.
- Improves performance.
- Executes multiple operations together.
- Commonly used for ETL, data migration, and batch processing.

# Supported Bulk Operations

The `bulk_write()` method supports the following operations.

| Operation | Description |
|-----------|-------------|
| `InsertOne()` | Insert a document |
| `UpdateOne()` | Update one document |
| `UpdateMany()` | Update multiple documents |
| `ReplaceOne()` | Replace a document |
| `DeleteOne()` | Delete one document |
| `DeleteMany()` | Delete multiple documents |

Each operation is executed in the order it appears in the list.

In [149]:
# =====================================
# PyMongo Command
# =====================================

from pymongo import (
    InsertOne,
    UpdateOne,
    DeleteOne,
    ReplaceOne
)

# Perform Multiple Operations

The following example performs four operations in a single request.

1. Insert a new customer.
2. Update the membership of an existing customer.
3. Replace an existing customer document.
4. Delete a customer.

Although these operations are different, MongoDB executes them together using **`bulk_write()`**.

In [150]:
# =====================================
# PyMongo Command
# =====================================

operations = [

    InsertOne(
        {
            "customer_id": 1012,
            "name": "Deepika Rao",
            "gender": "Female",
            "age": 29,
            "email": "deepika@example.com",
            "city": "Visakhapatnam",
            "state": "Andhra Pradesh",
            "membership": "Silver",
            "is_active": True
        }
    ),

    UpdateOne(
        {"customer_id": 1005},
        {
            "$set": {
                "membership": "Gold"
            }
        }
    ),

    ReplaceOne(
        {"customer_id": 1006},
        {
            "customer_id": 1006,
            "name": "Ananya Das",
            "gender": "Female",
            "age": 28,
            "email": "ananya@example.com",
            "city": "Kolkata",
            "state": "West Bengal",
            "membership": "Diamond",
            "is_active": True
        }
    ),

    DeleteOne(
        {"customer_id": 1010}
    )

]

result = customers.bulk_write(operations)

print(result.bulk_api_result)

{'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 1, 'nUpserted': 0, 'nMatched': 1, 'nModified': 1, 'nRemoved': 1, 'upserted': []}


In [151]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# MongoDB Shell does not provide a direct equivalent of PyMongo's bulk_write().
# Instead, similar functionality is available using db.collection.bulkWrite()
#
# Example:
#
# db.getSiblingDB('ecommerce').customers.bulkWrite([
#     { insertOne: { document: { customer_id: 1012, name: 'Deepika Rao', membership: 'Silver' } } },
#     { updateOne: { filter: { customer_id: 1005 }, update: { $set: { membership: 'Gold' } } } },
#     { replaceOne: { filter: { customer_id: 1006 }, replacement: { customer_id: 1006, name: 'Ananya Das', membership: 'Diamond' } } },
#     { deleteOne: { filter: { customer_id: 1010 } } }
# ])

# Verify the Changes

Let's verify that the bulk operations were successfully executed.

In [152]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find().sort("customer_id", 1)

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 29,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b88'),
 'age': 25,
 'city': 'Hyderabad',
 'customer_id': 1002,
 'email': 'priya.reddy@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-02-10',
 'membership': 'Gold',
 'name': 'Priya Reddy',
 'phone': '9876543211',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8a'),
 'age': 30,
 'city': 'Mumbai',
 'customer_id': 1004,
 'email': 'sneha.pate

# When Should Bulk Write Be Used?

Bulk Write is useful when processing a large number of documents.

Common use cases include:

- Importing CSV or Excel files
- Data migration
- Synchronizing databases
- Batch updates
- ETL (Extract, Transform, Load) pipelines

For a few operations, using `insert_one()` or `update_one()` is usually sufficient.

For hundreds or thousands of operations, **`bulk_write()`** provides better performance by reducing communication between the application and the MongoDB server.

# Query Execution Plans (`explain()`)

Whenever a query is executed, MongoDB determines the most efficient way to retrieve the requested documents.

This decision-making process is called the **Query Execution Plan**.

The **`explain()`** method displays how MongoDB executes a query.

It helps developers understand:

- Whether an index is being used.
- How many documents are scanned.
- Which execution plan MongoDB selected.
- How queries can be optimized.

The `explain()` method is commonly used for **performance tuning** and **query optimization**.

# Common Execution Stages

When analyzing an execution plan, the following stages are commonly encountered.

| Stage | Description |
|--------|-------------|
| `COLLSCAN` | Collection Scan (every document is scanned) |
| `IXSCAN` | Index Scan (MongoDB uses an index) |
| `FETCH` | Retrieves the actual documents after locating them through an index |
| `SORT` | Sorts the query results |
| `LIMIT` | Limits the number of returned documents |

Among these stages, **COLLSCAN** and **IXSCAN** are the most important for beginners.

# Explain a Query

The following example displays the execution plan for a query.

If no suitable index exists, MongoDB typically performs a **Collection Scan (COLLSCAN)**.

In [153]:
# =====================================
# PyMongo Command
# =====================================

plan = customers.find(
    {"city": "Hyderabad"}
).explain()

pprint(plan)

{'command': {'$db': 'ecommerce',
             'filter': {'city': 'Hyderabad'},
             'find': 'customers'},
 'executionStats': {'allPlansExecution': [],
                    'executionStages': {'advanced': 2,
                                        'alreadyHasObj': 0,
                                        'docsExamined': 2,
                                        'executionTimeMillisEstimate': 0,
                                        'inputStage': {'advanced': 2,
                                                       'direction': 'forward',
                                                       'dupsDropped': 0,
                                                       'dupsTested': 0,
                                                       'executionTimeMillisEstimate': 0,
                                                       'indexBounds': {'city': ['["Hyderabad", '
                                                                                '"Hyderabad"]'],
                

In [154]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({city:'Hyderabad'}).explain()"

{
  explainVersion: '1',
  queryPlanner: {
    namespace: 'ecommerce.customers',
    parsedQuery: { city: { '$eq': 'Hyderabad' } },
    indexFilterSet: false,
    queryHash: 'DE2E7A09',
    planCacheShapeHash: 'DE2E7A09',
    planCacheKey: '1D4DE2C1',
    optimizationTimeMillis: 0,
    maxIndexedOrSolutionsReached: false,
    maxIndexedAndSolutionsReached: false,
    maxScansToExplodeReached: false,
    prunedSimilarIndexes: false,
    winningPlan: {
      isCached: false,
      stage: 'FETCH',
      nss: 'ecommerce.customers',
      inputStage: {
        stage: 'IXSCAN',
        nss: 'ecommerce.customers',
        keyPattern: { city: 1, membership: 1 },
        indexName: 'city_1_membership_1',
        isMultiKey: false,
        multiKeyPaths: { city: [], membership: [] },
        isUnique: false,
        isSparse: false,
        isPartial: false,
        indexVersion: 2,
        direction: 'forward',
        indexBounds: {
          city: [ '["Hyderabad", "Hyderabad"]' ],
          m

# Understanding the Execution Plan

The output of `explain()` is a large document containing execution details.

For beginners, the most useful information is the **winning plan**.

The **winning plan** indicates the strategy chosen by MongoDB to execute the query.

In [155]:
# =====================================
# PyMongo Command
# =====================================

pprint(plan["queryPlanner"]["winningPlan"])

{'inputStage': {'direction': 'forward',
                'indexBounds': {'city': ['["Hyderabad", "Hyderabad"]'],
                                'membership': ['[MinKey, MaxKey]']},
                'indexName': 'city_1_membership_1',
                'indexVersion': 2,
                'isMultiKey': False,
                'isPartial': False,
                'isSparse': False,
                'isUnique': False,
                'keyPattern': {'city': 1, 'membership': 1},
                'multiKeyPaths': {'city': [], 'membership': []},
                'nss': 'ecommerce.customers',
                'stage': 'IXSCAN'},
 'isCached': False,
 'nss': 'ecommerce.customers',
 'stage': 'FETCH'}


# Collection Scan (COLLSCAN)

If the execution stage is:

```text
COLLSCAN
```

MongoDB scans every document in the collection.

Collection scans are acceptable for very small collections, but they become slower as the collection grows.

# Index Scan (IXSCAN)

If a suitable index exists, MongoDB may choose:

```text
IXSCAN
```

An Index Scan allows MongoDB to locate matching documents efficiently without scanning the entire collection.

For large collections, **IXSCAN** usually provides much better performance than **COLLSCAN**.

# Create an Index and Compare the Execution Plan

The following example creates an index on the **city** field and then displays the execution plan again.

Observe how MongoDB changes from **COLLSCAN** to **IXSCAN**.

In [156]:
# =====================================
# PyMongo Command
# =====================================

customers.create_index("city")

plan = customers.find(
    {"city": "Hyderabad"}
).explain()

pprint(plan["queryPlanner"]["winningPlan"])

{'inputStage': {'direction': 'forward',
                'indexBounds': {'city': ['["Hyderabad", "Hyderabad"]']},
                'indexName': 'city_1',
                'indexVersion': 2,
                'isMultiKey': False,
                'isPartial': False,
                'isSparse': False,
                'isUnique': False,
                'keyPattern': {'city': 1},
                'multiKeyPaths': {'city': []},
                'nss': 'ecommerce.customers',
                'stage': 'IXSCAN'},
 'isCached': False,
 'nss': 'ecommerce.customers',
 'stage': 'FETCH'}


In [157]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.createIndex({city:1}); db.getSiblingDB('ecommerce').customers.find({city:'Hyderabad'}).explain()"

{
  explainVersion: '1',
  queryPlanner: {
    namespace: 'ecommerce.customers',
    parsedQuery: { city: { '$eq': 'Hyderabad' } },
    indexFilterSet: false,
    queryHash: 'DE2E7A09',
    planCacheShapeHash: 'DE2E7A09',
    planCacheKey: '600ED859',
    optimizationTimeMillis: 0,
    maxIndexedOrSolutionsReached: false,
    maxIndexedAndSolutionsReached: false,
    maxScansToExplodeReached: false,
    prunedSimilarIndexes: false,
    winningPlan: {
      isCached: false,
      stage: 'FETCH',
      nss: 'ecommerce.customers',
      inputStage: {
        stage: 'IXSCAN',
        nss: 'ecommerce.customers',
        keyPattern: { city: 1 },
        indexName: 'city_1',
        isMultiKey: false,
        multiKeyPaths: { city: [] },
        isUnique: false,
        isSparse: false,
        isPartial: false,
        indexVersion: 2,
        direction: 'forward',
        indexBounds: { city: [ '["Hyderabad", "Hyderabad"]' ] }
      }
    },
    rejectedPlans: [
      {
        isCached: fa

# Why Use `explain()`?

The `explain()` method is useful when:

- Queries become slow.
- Collections contain millions of documents.
- Multiple indexes exist.
- Query performance needs to be optimized.

Database administrators and developers frequently use `explain()` to understand how MongoDB executes queries and to determine whether additional indexes are required.

# MongoDB Atlas

**MongoDB Atlas** is the official cloud database service provided by MongoDB.

Instead of installing and maintaining MongoDB on a local machine, Atlas hosts MongoDB databases in the cloud.

Atlas provides:

- Free cloud clusters
- Automatic backups
- Replica Sets
- High Availability
- Monitoring
- Security
- Automatic Scaling

Atlas is widely used in production applications because it eliminates the need to manage database servers manually.

# Local MongoDB vs MongoDB Atlas

| Local MongoDB | MongoDB Atlas |
|---------------|---------------|
| Runs on your computer | Runs in the cloud |
| Manual installation | No installation required |
| Manual backups | Automatic backups |
| Standalone by default | Replica Set by default |
| Limited accessibility | Accessible from anywhere |
| Suitable for development | Suitable for production |

During development, developers usually work with a local MongoDB server.

For deployment and production systems, MongoDB Atlas is the preferred choice.

# Steps to Create a Free MongoDB Atlas Cluster

1. Create an account on MongoDB Atlas.
2. Create a new project.
3. Create a Free Tier (M0) Cluster.
4. Create a Database User.
5. Configure Network Access (IP Whitelist).
6. Obtain the Connection String.
7. Connect using:
   - MongoDB Compass
   - mongosh
   - PyMongo

# Example Connection String

A MongoDB Atlas connection string looks similar to the following.

```text
mongodb+srv://<username>:<password>@cluster0.xxxxx.mongodb.net/?retryWrites=true&w=majority
```

Replace:

- `<username>` with your Atlas database username.
- `<password>` with your Atlas database password.

Never hardcode usernames and passwords in production applications.

Instead, store them securely using environment variables or configuration files.

In [178]:
# =====================================
# PyMongo Command
# =====================================

import os
from pymongo import MongoClient
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Read the Atlas connection string
ATLAS_URI = os.getenv("ATLAS_URI")

# Create the MongoDB client
atlas_client = MongoClient(ATLAS_URI)

# Verify the connection
atlas_client.admin.command("ping")

print("Successfully Connected to MongoDB Atlas")

Successfully Connected to MongoDB Atlas


In [179]:
# =====================================
# mongosh Command
# =====================================

!mongosh "{ATLAS_URI}" --eval "db.adminCommand({ ping: 1 })"

{ ok: 1 }


# List Databases in MongoDB Atlas

Once connected, MongoDB Atlas behaves exactly like a local MongoDB server.

The following command lists all available databases.

In [180]:
# =====================================
# PyMongo Command
# =====================================

atlas_client.list_database_names()

['admin', 'local']

In [174]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh "{ATLAS_URI}" --eval "show dbs"

admin  0 B
local  0 B


# Working with MongoDB Atlas

After connecting, all MongoDB operations remain the same.

For example, selecting a database:

```python
db = atlas_client["ecommerce"]
```

Creating a collection:

```python
customers = db["customers"]
```

Performing CRUD operations:

- Insert
- Find
- Update
- Delete
- Aggregate

No changes are required in your application code except replacing the connection string.

# Why Learn MongoDB Atlas?

MongoDB Atlas is widely used in:

- Web Applications
- Mobile Applications
- AI Applications
- Cloud-Native Systems
- Data Analytics Platforms
- Enterprise Applications

Understanding Atlas prepares students for real-world MongoDB deployments and cloud-based database management.

# Replica Sets

A **Replica Set** is a group of MongoDB servers that maintain the same data.

Each server stores a copy of the database, providing:

- High Availability
- Fault Tolerance
- Automatic Failover
- Data Redundancy

Replica Sets ensure that the database remains available even if one of the servers fails.

# Why Do We Need Replica Sets?

Imagine a MongoDB server running on a single computer.

If that computer fails due to:

- Hardware failure
- Power outage
- Network failure
- Operating system crash

the database becomes unavailable.

To avoid this problem, MongoDB stores multiple copies of the database on different servers.

This group of synchronized servers is called a **Replica Set**.

# Components of a Replica Set

A Replica Set usually consists of three types of members.

| Member | Description |
|---------|-------------|
| Primary | Receives all write operations |
| Secondary | Maintains a copy of the Primary and can serve read operations (if configured) |
| Arbiter | Participates only in elections and does not store data |

The most common deployment consists of:

- One Primary
- Two Secondary nodes

# Replica Set Architecture

```mermaid
flowchart LR

A["Application"]

A --> P["Primary"]

P --> S1["Secondary 1"]

P --> S2["Secondary 2"]

P -. Replication .-> S1
P -. Replication .-> S2
```

All write operations are sent to the **Primary** node.

The Primary continuously replicates data to the Secondary nodes.

# Automatic Failover

If the Primary server becomes unavailable, MongoDB automatically elects one of the Secondary nodes as the new Primary.

This process is called **Automatic Failover**.

```mermaid
flowchart LR

P["Primary ❌"]

S1["Secondary 1"]

S2["Secondary 2"]

S1 ==> NP["New Primary"]

S2 --> NP
```

This process usually completes within a few seconds, allowing applications to continue operating with minimal interruption.

# Read and Write Operations

In a Replica Set:

- All **write operations** are sent to the Primary node.
- Secondary nodes continuously replicate data from the Primary.
- Read operations are typically served by the Primary, but they can also be directed to Secondary nodes using Read Preferences.

This architecture provides:

- Data consistency
- High availability
- Improved read scalability

# Replica Sets in MongoDB Atlas

Every MongoDB Atlas cluster is configured as a **Replica Set** by default.

This means:

- No manual configuration is required.
- Automatic failover is already enabled.
- High availability is built in.
- Transactions are supported.

When you create a free **M0 Atlas Cluster**, MongoDB automatically configures a Replica Set for you.

In [181]:
# =====================================
# PyMongo Command
# =====================================

atlas_client.admin.command("hello")

{'topologyVersion': {'processId': ObjectId('6a5cfe639e1fde714b3a17bf'),
  'counter': 6},
 'hosts': ['ac-vycbybn-shard-00-00.xtvjwjv.mongodb.net:27017',
  'ac-vycbybn-shard-00-01.xtvjwjv.mongodb.net:27017',
  'ac-vycbybn-shard-00-02.xtvjwjv.mongodb.net:27017'],
 'setName': 'atlas-12209n-shard-0',
 'setVersion': 193,
 'isWritablePrimary': True,
 'secondary': False,
 'primary': 'ac-vycbybn-shard-00-01.xtvjwjv.mongodb.net:27017',
 'tags': {'diskState': 'READY',
  'availabilityZone': 'aps1-az3',
  'provider': 'AWS',
  'region': 'AP_SOUTH_1',
  'cacheState': 'READY',
  'workloadType': 'OPERATIONAL',
  'nodeType': 'ELECTABLE'},
 'me': 'ac-vycbybn-shard-00-01.xtvjwjv.mongodb.net:27017',
 'electionId': ObjectId('7fffffff00000000000002b9'),
 'lastWrite': {'opTime': {'ts': Timestamp(1784493525, 9), 't': 697},
  'lastWriteDate': datetime.datetime(2026, 7, 19, 20, 38, 45),
  'majorityOpTime': {'ts': Timestamp(1784493525, 9), 't': 697},
  'majorityWriteDate': datetime.datetime(2026, 7, 19, 20, 38, 4

In [182]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh "{ATLAS_URI}" --eval "db.hello()"

{
  topologyVersion: {
    processId: ObjectId('6a5cfe639e1fde714b3a17bf'),
    counter: Long('6')
  },
  hosts: [
    'ac-vycbybn-shard-00-00.xtvjwjv.mongodb.net:27017',
    'ac-vycbybn-shard-00-01.xtvjwjv.mongodb.net:27017',
    'ac-vycbybn-shard-00-02.xtvjwjv.mongodb.net:27017'
  ],
  setName: 'atlas-12209n-shard-0',
  setVersion: 193,
  isWritablePrimary: true,
  secondary: false,
  primary: 'ac-vycbybn-shard-00-01.xtvjwjv.mongodb.net:27017',
  tags: {
    cacheState: 'READY',
    region: 'AP_SOUTH_1',
    provider: 'AWS',
    availabilityZone: 'aps1-az3',
    diskState: 'READY',
    nodeType: 'ELECTABLE',
    workloadType: 'OPERATIONAL'
  },
  me: 'ac-vycbybn-shard-00-01.xtvjwjv.mongodb.net:27017',
  electionId: ObjectId('7fffffff00000000000002b9'),
  lastWrite: {
    opTime: { ts: Timestamp({ t: 1784493532, i: 1 }), t: Long('697') },
    lastWriteDate: ISODate('2026-07-19T20:38:52.000Z'),
    majorityOpTime: { ts: Timestamp({ t: 1784493532, i: 1 }), t: Long('697') },
    majority

# Why Are Replica Sets Important?

Replica Sets provide several important benefits:

- High Availability
- Automatic Failover
- Data Redundancy
- Improved Reliability
- Foundation for Distributed Systems
- Required for Multi-Document Transactions

Without a Replica Set, MongoDB cannot provide fault tolerance or support distributed transactions.

# Transactions

A **Transaction** is a sequence of one or more database operations that are treated as a **single unit of work**.

Either:

- All operations are completed successfully (**Commit**), or
- None of the operations are applied (**Rollback / Abort**).

Transactions help maintain data consistency when multiple operations must succeed or fail together.

# Why Do We Need Transactions?

Consider a banking application where ₹500 is transferred from **Account A** to **Account B**.

The transfer consists of two operations:

1. Deduct ₹500 from Account A.
2. Add ₹500 to Account B.

Suppose the server crashes after deducting money from Account A but before adding it to Account B.

Without a transaction:

- Account A loses ₹500.
- Account B never receives the money.

The database becomes inconsistent.

A transaction ensures that **both operations succeed or both are cancelled**.

# Money Transfer Example

```mermaid
flowchart LR

A["Account A<br/>₹5000"]

B["Account B<br/>₹2000"]

A -- "- ₹500" --> T["Transaction"]

T -- "+ ₹500" --> B

style T fill:#c8f7c5
```

If any step fails, MongoDB rolls back the entire transaction.

# ACID Properties

MongoDB transactions follow the **ACID** properties.

| Property | Description |
|----------|-------------|
| Atomicity | All operations succeed or none are applied |
| Consistency | Database remains in a valid state |
| Isolation | Concurrent transactions do not interfere with each other |
| Durability | Once committed, changes are permanently stored |

These properties ensure reliable and consistent database operations.

# Transactions and Replica Sets

Multi-document transactions require a **Replica Set** or a **Sharded Cluster**.

A standalone MongoDB server does **not** support multi-document transactions.

Since MongoDB Atlas clusters are Replica Sets by default, transactions work without additional configuration.

# Sample Data

Assume the following documents exist in the **accounts** collection.

```json
[
    {
        "_id": 1,
        "name": "Alice",
        "balance": 5000
    },
    {
        "_id": 2,
        "name": "Bob",
        "balance": 2000
    }
]
```

# Money Transfer Using a Transaction

The following example transfers ₹500 from Alice to Bob.

If any operation fails, the entire transaction is aborted.

In [183]:
# =====================================
# PyMongo Command
# =====================================

accounts = atlas_client["bank"]["accounts"]

with atlas_client.start_session() as session:
    try:
        with session.start_transaction():

            accounts.update_one(
                {"_id": 1},
                {"$inc": {"balance": -500}},
                session=session
            )

            accounts.update_one(
                {"_id": 2},
                {"$inc": {"balance": 500}},
                session=session
            )

        print("Transaction Committed Successfully")

    except Exception as e:
        print("Transaction Aborted")
        print(e)

Transaction Committed Successfully


In [197]:
!mongosh "{ATLAS_URI}" --eval "const session=db.getMongo().startSession(); const bank=session.getDatabase('bank'); try{ session.startTransaction(); bank.accounts.updateOne({_id:1},{$inc:{balance:-500}}); bank.accounts.updateOne({_id:2},{$inc:{balance:500}}); session.commitTransaction(); print('✅ Transaction Committed Successfully'); } catch(e){ session.abortTransaction(); print('❌ Transaction Aborted'); print(e); } finally{ session.endSession(); }"

❌ Transaction Aborted
MongoServerError: This MongoDB deployment does not support retryable writes. Please add retryWrites=false to your connection string.
    at topology (eval at <anonymous> (node:lib-boxednode/mongosh:103:20), <anonymous>:3:7267874)
    at process.processTicksAndRejections (node:internal/process/task_queues:104:5)
    at async t.executeOperation (eval at <anonymous> (node:lib-boxednode/mongosh:103:20), <anonymous>:3:7266633)
    at async Collection.updateOne (eval at <anonymous> (node:lib-boxednode/mongosh:103:20), <anonymous>:3:7106756)
    at async Proxy.updateOne (eval at <anonymous> (node:lib-boxednode/mongosh:103:20), <anonymous>:140:618404)
    at async Proxy.updateOne (eval at <anonymous> (node:lib-boxednode/mongosh:103:20), <anonymous>:140:705008)
    at async Proxy.eval (eval at <anonymous> (node:lib-boxednode/mongosh:103:20), <anonymous>:140:707114)
    at async Proxy.eval (eval at <anonymous> (node:lib-boxednode/mongosh:103:20), <anonymous>:140:707505)
   

In [199]:
!mongosh "{ATLAS_URI}" --eval "db.runCommand({connectionStatus:1})"

{
  authInfo: { authenticatedUsers: [], authenticatedUserRoles: [] },
  uuid: UUID('cd08986b-84ca-4f78-a249-325d44523ddc'),
  ok: 1
}


# Transaction Workflow

A transaction generally follows these steps:

1. Start a Session.
2. Start a Transaction.
3. Perform one or more database operations.
4. Commit the Transaction if all operations succeed.
5. Abort the Transaction if any operation fails.

```mermaid
flowchart LR

A["Start Session"]
--> B["Start Transaction"]
--> C["Execute Operations"]
--> D{"Success?"}

D -->|Yes| E["Commit Transaction"]

D -->|No| F["Abort Transaction"]
```

# When Should Transactions Be Used?

Transactions are useful when multiple related operations must succeed together.

Common use cases include:

- Banking and financial systems
- Online payment processing
- E-commerce order placement
- Inventory management
- Airline and railway booking systems
- Hospital management systems

If only a single document is being modified, MongoDB already guarantees **atomicity** for that document, so a transaction is usually unnecessary.

# Best Practices

- Keep transactions short.
- Avoid long-running transactions.
- Modify only the required documents.
- Commit transactions as soon as possible.
- Use transactions only when multiple operations must succeed together.

For single-document updates, normal CRUD operations are generally sufficient.